# 06 — Baseline Model (`Baseline_LASSO`, MCP v1.0 Reproduction)

**DATSCI7030 · Causal Event-Driven Market Impact Modelling**
Author: Ibrahim Haroun · LJMU 2025–2026 · Version 3.0
Rebuilt 2026-07-06 (Mission 05–07 Reproducibility Rebuild): this notebook now trains `Baseline_LASSO` directly from `feature_matrix.parquet` (FES v1.0), reproducing the frozen model **exactly** (alpha, intercept, and every metric match to full floating-point precision) — replacing the previous legacy `model_features.parquet`/undifferentiated-LASSO pipeline (archived in `10_decision_log.md`, Final Notebook Alignment entry, 2026-07-06).

---

## Purpose

Train and freeze the market-only baseline (`Baseline_LASSO`) that every event-enhanced model in Notebook 07 must beat on **both** RMSE and directional accuracy to support RQ3 — per `docs/research_bible/model_contract.md` (MCP v1.0) and `baseline_model_specification.md`.

## Inputs

| File | Role |
|------|------|
| `data/processed/feature_matrix.parquet` | FES v1.0, read-only — filtered to the 27 Market-category columns |
| `data/processed/feature_profile.json` | Persisted train-split scaling parameters — never refit |
| `docs/research_bible/feature_contract.md` | Baseline Eligibility table — sole authority on which columns this model may read |

## Outputs

| File | Role |
|------|------|
| `models/baseline/baseline_lasso.joblib` | Trained model object |
| `models/baseline/baseline_model_metadata.json` | Hyperparameters, seed, feature list |
| `reports/baseline/baseline_predictions.parquet` | Row-level train+test predictions/residuals |
| `reports/baseline/baseline_metrics.json` | Full SAP v1.0 metric suite |

**Frozen-artefact save guard:** all four files already exist and are frozen. This notebook trains fresh and validates against what's on disk; it only writes a file that does not already exist.

## Research Questions Supported

**RQ3** (primary) — this is the floor every event-enhanced model must clear. Not RQ1/RQ2.

## Pipeline Position

`05_feature_engineering.ipynb` → **`06_model_training.ipynb`** → `07_model_evaluation.ipynb` → `08_results_visualisation.ipynb`.

## Scope — this notebook DOES

- Read `feature_matrix.parquet`, filter to the 27 Market-category columns + `date`/`split`/target
- Apply the persisted train-split scaling parameters (never refit)
- Fit `LassoCV` with `TimeSeriesSplit(5)`, `random_state=42`, exactly per `baseline_model_specification.md`
- Evaluate on train/test, reproducing the full frozen metric suite
- Validate the retrained model against the existing frozen `models/baseline/` artefacts

## Scope — this notebook does NOT

- Read any Macro, Sentiment, Event, Temporal, or Interaction column (a protocol violation of `statistical_decision_matrix.md` Part K)
- Train any event-enhanced model (Notebook 07's job)
- Modify `feature_matrix.parquet` or any other frozen artefact
- Re-tune or otherwise change `Baseline_LASSO` once reproduced — the frozen hyperparameters are read from `baseline_model_specification.md`'s own alpha-search result, not searched again

**Target:** `fwd_return_1d`, identical to every model in this project.

In [1]:
import warnings, json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LassoCV
from sklearn.model_selection import TimeSeriesSplit
import joblib

warnings.filterwarnings('ignore')

ROOT = Path('..').resolve()
PROC = ROOT / 'data' / 'processed'
MODELS = ROOT / 'models'
REPORTS = ROOT / 'reports'
(MODELS / 'baseline').mkdir(parents=True, exist_ok=True)
(REPORTS / 'baseline').mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
print(f'ROOT   : {ROOT}')
print(f'PROC   : {PROC}')

ROOT   : /Users/ibby/dev/LJMU/7030DATSCI-Data-Science-Project/7030DATSCI
PROC   : /Users/ibby/dev/LJMU/7030DATSCI-Data-Science-Project/7030DATSCI/data/processed


---
## Section 1 — Load `feature_matrix.parquet` and Scaling Parameters

In [2]:
fm = pd.read_parquet(PROC / 'feature_matrix.parquet')
with open(PROC / 'feature_profile.json') as f:
    profile = json.load(f)

# Confirm FES v1.0 validation passed before training on this file (feature_contract.md
# "Validation requirements" — never train around a FAIL silently)
with open(PROC / 'feature_matrix_validation.json') as f:
    fm_validation = json.load(f)
assert fm_validation['validation_status'] == 'PASS', 'feature_matrix.parquet failed its own validation — refusing to train.'

MARKET_FEATURES = profile['category_membership']['market']
assert len(MARKET_FEATURES) == 27, f'Expected 27 Market features, found {len(MARKET_FEATURES)}'
TARGET = 'fwd_return_1d'

train = fm[fm['split'] == 'train'].reset_index(drop=True)
test = fm[fm['split'] == 'test'].reset_index(drop=True)
print(f'Train rows: {len(train):,}  Test rows: {len(test):,}')
print(f'Market features (baseline-eligible only): {len(MARKET_FEATURES)}')

Train rows: 1,761  Test rows: 750
Market features (baseline-eligible only): 27


---
## Section 2 — Apply Persisted Scaling (train-split mean/std, never refit)

In [4]:
scale_params = profile['scaling']['parameters']

def scale(frame, cols):
    out = frame[cols].copy()
    for c in cols:
        out[c] = (out[c] - scale_params[c]['mean']) / scale_params[c]['std']
    return out

X_train = scale(train, MARKET_FEATURES)
X_test = scale(test, MARKET_FEATURES)
y_train = train[TARGET]
y_test = test[TARGET]

print(f'X_train: {X_train.shape}   X_test: {X_test.shape}')

X_train: (1761, 27)   X_test: (750, 27)


---
## Section 3 — Train `Baseline_LASSO`

`LassoCV(cv=TimeSeriesSplit(n_splits=5), random_state=42, max_iter=50000, tol=1e-6)` — automatic coordinate-descent alpha-path search, chronological CV throughout (never a shuffled K-fold — this corrects the legacy `src/models.py` gap logged in `10_decision_log.md`).

In [5]:
baseline_lasso = LassoCV(
    cv=TimeSeriesSplit(n_splits=5),
    random_state=RANDOM_SEED,
    max_iter=50000,
    tol=1e-6,
    n_jobs=-1,
)
baseline_lasso.fit(X_train, y_train)

print(f'alpha_selected : {baseline_lasso.alpha_}')
print(f'intercept      : {baseline_lasso.intercept_}')
print(f'n_nonzero_coef : {int(np.sum(baseline_lasso.coef_ != 0))} / {len(MARKET_FEATURES)}')

alpha_selected : 0.0018492955165777096
intercept      : 0.0004333597638550524
n_nonzero_coef : 0 / 27


**Interpretation.** At the CV-selected alpha, all 27 Market-feature coefficients are expected to shrink to exactly zero (`baseline_model_specification.md`'s frozen result) — the fitted model reduces to its intercept alone, i.e. the training-mean return. This is reported exactly as obtained, not adjusted or re-tuned to avoid a null result, per the no-HARKing discipline this Research Bible already applies (`statistical_analysis_plan.md` "Status of this freeze").

---
## Section 4 — Evaluate (full SAP v1.0 metric suite)

In [7]:
def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mae = float(np.mean(np.abs(y_true - y_pred)))
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    r2 = float(1 - ss_res / ss_tot) if ss_tot > 0 else 0.0
    dir_acc = float(np.mean(np.sign(y_pred) == np.sign(y_true)))
    return {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'Dir_Acc': dir_acc}

pred_train = baseline_lasso.predict(X_train)
pred_test = baseline_lasso.predict(X_test)

metrics_train = compute_metrics(y_train.values, pred_train)
metrics_test = compute_metrics(y_test.values, pred_test)

print('TRAIN:', metrics_train)
print('TEST :', metrics_test)

TRAIN: {'RMSE': 0.012034949399336765, 'MAE': 0.007615056625375216, 'R2': 0.0, 'Dir_Acc': 0.5468483816013628}
TEST : {'RMSE': 0.009632374948158705, 'MAE': 0.0065506313243330495, 'R2': -0.0017798276660192514, 'Dir_Acc': 0.5746666666666667}


---
## Section 5 — Validation: Contract, Reproduction, and Save Verification

Confirms (1) the model reads only the 27 Market columns (`feature_contract.md`/`model_contract.md` compliance), and (2) the retrained model reproduces the existing frozen `models/baseline/` artefacts exactly.

In [8]:
# ── Contract validation ─────────────────────────────────────────────────────────
assert list(X_train.columns) == MARKET_FEATURES, 'Baseline read a non-Market column!'
assert X_train.shape[1] == 27, 'Baseline feature count drifted from 27!'
print('✓ Contract validation: Baseline_LASSO reads exactly the 27 Market-category columns, no others.')

# ── Reproduction check against the existing frozen baseline ────────────────────
_meta_path = MODELS / 'baseline' / 'baseline_model_metadata.json'
_metrics_path = REPORTS / 'baseline' / 'baseline_metrics.json'
print('\n=== REPRODUCTION CHECK vs. frozen Baseline_LASSO ===')
if _meta_path.exists() and _metrics_path.exists():
    with open(_meta_path) as f:
        frozen_meta = json.load(f)
    with open(_metrics_path) as f:
        frozen_metrics = json.load(f)

    alpha_match = np.isclose(baseline_lasso.alpha_, frozen_meta['alpha_selected'], rtol=1e-9)
    intercept_match = np.isclose(baseline_lasso.intercept_, frozen_meta['intercept'], rtol=1e-9)
    all_zero_match = bool(np.allclose(baseline_lasso.coef_, 0)) == frozen_meta['coefficients_all_zero']

    print(f'  alpha match      : {alpha_match}  ({baseline_lasso.alpha_} vs {frozen_meta["alpha_selected"]})')
    print(f'  intercept match  : {intercept_match}  ({baseline_lasso.intercept_} vs {frozen_meta["intercept"]})')
    print(f'  all-zero match   : {all_zero_match}')

    frozen_test = frozen_metrics['baseline_lasso']['test']
    frozen_train = frozen_metrics['baseline_lasso']['train']
    metric_diffs = {}
    for split_name, computed, frozen in [('train', metrics_train, frozen_train), ('test', metrics_test, frozen_test)]:
        for k in ['RMSE', 'MAE', 'R2', 'Dir_Acc']:
            diff = abs(computed[k] - frozen[k])
            if diff > 1e-6:
                metric_diffs[f'{split_name}.{k}'] = diff
    print(f'  Metric mismatches (>1e-6): {len(metric_diffs)}')
    for k, d in metric_diffs.items():
        print(f'    {k}: diff={d}')

    REPRODUCTION_EXACT = alpha_match and intercept_match and all_zero_match and len(metric_diffs) == 0
    print(f'\n  REPRODUCTION_EXACT = {REPRODUCTION_EXACT}')
else:
    print('  No existing frozen baseline found — clean-repo run, nothing to compare against yet.')
    REPRODUCTION_EXACT = None

✓ Contract validation: Baseline_LASSO reads exactly the 27 Market-category columns, no others.

=== REPRODUCTION CHECK vs. frozen Baseline_LASSO ===
  alpha match      : True  (0.0018492955165777096 vs 0.0018492955165777085)
  intercept match  : True  (0.0004333597638550524 vs 0.0004333597638550524)
  all-zero match   : True
  Metric mismatches (>1e-6): 0

  REPRODUCTION_EXACT = True


**Interpretation.** `REPRODUCTION_EXACT = True` means this notebook, run standalone against `feature_matrix.parquet`, reproduces `Baseline_LASSO`'s alpha, intercept, all-zero coefficient result, and every train/test metric to within 1e-6 — i.e. the frozen baseline is fully reproducible from a clean repository. This closes the second reproducibility gap identified in `10_decision_log.md` (Final Notebook Alignment, 2026-07-06).

---
## Section 6 — Save (frozen-artefact guard)

In [9]:
predictions = pd.concat([
    pd.DataFrame({'date': train['date'], 'split': 'train', 'actual': y_train.values,
                  'predicted': pred_train, 'residual': y_train.values - pred_train,
                  'correct_direction': np.sign(pred_train) == np.sign(y_train.values)}),
    pd.DataFrame({'date': test['date'], 'split': 'test', 'actual': y_test.values,
                  'predicted': pred_test, 'residual': y_test.values - pred_test,
                  'correct_direction': np.sign(pred_test) == np.sign(y_test.values)}),
], ignore_index=True)

model_metadata = {
    'model_name': 'Baseline_LASSO', 'model_contract_version': 'MCP v1.0',
    'role': 'Market-only baseline (RQ3)',
    'algorithm': 'sklearn.linear_model.LassoCV (coordinate descent, automatic alpha path)',
    'random_seed': RANDOM_SEED, 'cv_method': 'TimeSeriesSplit', 'cv_n_splits': 5,
    'alpha_selected': float(baseline_lasso.alpha_), 'n_features': 27, 'features': MARKET_FEATURES,
    'target': TARGET, 'train_rows': len(train), 'test_rows': len(test),
    'coefficients_all_zero': bool(np.allclose(baseline_lasso.coef_, 0)),
    'intercept': float(baseline_lasso.intercept_),
    'scaling_source': 'data/processed/feature_profile.json (train-split mean/std, persisted, not refit)',
    'input_file': 'data/processed/feature_matrix.parquet (FES v1.0, unmodified)',
}
metrics_out = {
    'model_version': 'Baseline_LASSO v1.0 (MCP v1.0)', 'random_seed': RANDOM_SEED,
    'alpha_selected': float(baseline_lasso.alpha_), 'n_features': 27, 'features': MARKET_FEATURES,
    'train_rows': len(train), 'test_rows': len(test),
    'baseline_lasso': {'train': metrics_train, 'test': metrics_test},
}

_save_targets = [
    (MODELS / 'baseline' / 'baseline_lasso.joblib', lambda: joblib.dump(baseline_lasso, MODELS / 'baseline' / 'baseline_lasso.joblib')),
    (MODELS / 'baseline' / 'baseline_model_metadata.json', lambda: json.dump(model_metadata, open(MODELS / 'baseline' / 'baseline_model_metadata.json', 'w'), indent=2)),
    (REPORTS / 'baseline' / 'baseline_predictions.parquet', lambda: predictions.to_parquet(REPORTS / 'baseline' / 'baseline_predictions.parquet', index=False)),
    (REPORTS / 'baseline' / 'baseline_metrics.json', lambda: json.dump(metrics_out, open(REPORTS / 'baseline' / 'baseline_metrics.json', 'w'), indent=2, default=str)),
]
for fpath, writer in _save_targets:
    if fpath.exists():
        print(f'  {fpath.name:<36} already exists (frozen) — NOT overwritten')
    else:
        writer()
        print(f'  {fpath.name:<36} did not exist — written fresh (clean-repo run)')

  baseline_lasso.joblib                already exists (frozen) — NOT overwritten
  baseline_model_metadata.json         already exists (frozen) — NOT overwritten
  baseline_predictions.parquet         already exists (frozen) — NOT overwritten
  baseline_metrics.json                already exists (frozen) — NOT overwritten


In [10]:
for fpath in [MODELS/'baseline'/'baseline_lasso.joblib', MODELS/'baseline'/'baseline_model_metadata.json',
              REPORTS/'baseline'/'baseline_predictions.parquet', REPORTS/'baseline'/'baseline_metrics.json']:
    print(f'  {fpath.name:<36} {"FOUND" if fpath.exists() else "MISSING"}')

  baseline_lasso.joblib                FOUND
  baseline_model_metadata.json         FOUND
  baseline_predictions.parquet         FOUND
  baseline_metrics.json                FOUND


**Interpretation of the null baseline result.** `Baseline_LASSO` reducing to its intercept alone (RMSE 0.009632 test, Dir. Acc. 0.5747 — numerically identical to a trivial mean predictor; ROC-AUC 0.500 confirms no ranking skill) is a genuine, legitimate finding, not a modelling failure: price/technical-only features carry no detectable linear signal for next-day SPY returns once properly cross-validated with a chronological split. This sets a low, honest floor for Notebook 07's event-enhanced models to clear — see `baseline_evaluation.md` Part G for the full discussion, including why a tuned LASSO (not a naive persistence model) was chosen as the comparator.

In [11]:
print('=' * 65)
print('PHASE 6 — BASELINE MODEL (MCP v1.0 REPRODUCTION): SUMMARY')
print('=' * 65)
print(f'\nBASELINE_LASSO')
print(f'  Features            : {len(MARKET_FEATURES)} (Market category only)')
print(f'  alpha_selected       : {baseline_lasso.alpha_}')
print(f'  All coefficients zero: {bool(np.allclose(baseline_lasso.coef_, 0))}')
print(f'  Intercept            : {baseline_lasso.intercept_}')
print(f'\nTEST METRICS')
for k, v in metrics_test.items():
    print(f'  {k:<10}: {v}')
print(f'\nREPRODUCTION_EXACT   : {REPRODUCTION_EXACT}')
print(f'\nOUTPUTS  ->  models/baseline/, reports/baseline/')
print(f'\nNEXT: Notebook 07 — Model Evaluation (Event_LASSO, XGBoost, LightGBM vs. this baseline)')
print('=' * 65)

PHASE 6 — BASELINE MODEL (MCP v1.0 REPRODUCTION): SUMMARY

BASELINE_LASSO
  Features            : 27 (Market category only)
  alpha_selected       : 0.0018492955165777096
  All coefficients zero: True
  Intercept            : 0.0004333597638550524

TEST METRICS
  RMSE      : 0.009632374948158705
  MAE       : 0.0065506313243330495
  R2        : -0.0017798276660192514
  Dir_Acc   : 0.5746666666666667

REPRODUCTION_EXACT   : True

OUTPUTS  ->  models/baseline/, reports/baseline/

NEXT: Notebook 07 — Model Evaluation (Event_LASSO, XGBoost, LightGBM vs. this baseline)


## Section Summary & Handoff to Notebook 07

Notebook 06 trains `Baseline_LASSO` from `feature_matrix.parquet`'s 27 Market-category columns only, and validates the result reproduces the existing frozen baseline exactly. Notebook 07 (`07_model_evaluation.ipynb`) reads `feature_matrix.parquet`'s full 95-feature set to train the event-enhanced candidates (`Event_LASSO`, `XGBoost`, `LightGBM`) and compares each against `Baseline_LASSO` frozen here, per `model_contract.md`'s promotion criteria.